# Notebook 03 — Modelado y Selección
**Proyecto:** Sistema de Clasificación de Riesgo Académico Estudiantil  
**Tarea:** Clasificación multiclase (A/B/C/D/F)  
**Métrica principal:** F1-score macro

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Entorno listo.')

## 1. Carga y preparación (reproducible)

In [ ]:
df = pd.read_csv('../data/processed/dataset_limpio.csv')
X = df.drop(columns=['GradeClass'])
y = df['GradeClass']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cont_cols = ['StudyTimeWeekly', 'Absences', 'Age']
scaler = StandardScaler()
X_train_sc = X_train.copy()
X_test_sc = X_test.copy()
X_train_sc[cont_cols] = scaler.fit_transform(X_train[cont_cols])
X_test_sc[cont_cols] = scaler.transform(X_test[cont_cols])

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print('Variables de entrada:', list(X.columns))

## 2. Entrenamiento y comparación de tres algoritmos

In [ ]:
label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'F'}

models = {
    'RandomForestClassifier':      RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'GradientBoostingClassifier':  GradientBoostingClassifier(n_estimators=150, random_state=42),
    'LogisticRegression':          LogisticRegression(max_iter=1000, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    results[name] = {
        'f1_macro':        round(f1_score(y_test, y_pred, average='macro'), 4),
        'accuracy':        round(accuracy_score(y_test, y_pred), 4),
        'precision_macro': round(precision_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'recall_macro':    round(recall_score(y_test, y_pred, average='macro'), 4),
    }

df_results = pd.DataFrame(results).T.sort_values('f1_macro', ascending=False)
print('--- Comparación de modelos (conjunto de PRUEBA) ---')
print(df_results.to_string())

In [ ]:
# Visualización comparativa
fig, ax = plt.subplots(figsize=(9, 4))
metrics = ['f1_macro', 'accuracy', 'precision_macro', 'recall_macro']
x = np.arange(len(metrics))
width = 0.25
colors = ['#3498db', '#2ecc71', '#e67e22']
for i, (name, row) in enumerate(df_results.iterrows()):
    vals = [row[m] for m in metrics]
    ax.bar(x + i*width, vals, width, label=name.replace('Classifier','').replace('Regression','Reg'), color=colors[i], alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(['F1 Macro', 'Accuracy', 'Precision', 'Recall'])
ax.set_ylim(0, 0.7)
ax.set_title('Comparación de Modelos — Conjunto de Prueba')
ax.legend(loc='upper right')
ax.set_ylabel('Valor de métrica')
plt.tight_layout()
plt.savefig('../docs/comparacion_modelos.png', dpi=120)
plt.show()

## 3. Justificación de la selección del modelo

**Modelo seleccionado: LogisticRegression**

Aunque los tres modelos presentan métricas similares en este dataset (F1 macro entre 0.43 y 0.50), se selecciona LogisticRegression por:

1. **Mayor F1-score macro** sobre el conjunto de prueba (0.4961) — métrica definida en la Entrega 1.
2. **Interpretabilidad**: los coeficientes del modelo permiten explicar directamente qué variables influyen más en la predicción, lo que es relevante para coordinadores académicos (usuario final definido).
3. **Menor riesgo de sobreajuste**: los modelos de árbol (RF, GBM) mostraron un gap mayor entre entrenamiento y prueba en experimentos internos.
4. **Velocidad de predicción**: crítica para un dashboard en tiempo real.

**Nota sobre las métricas:** F1 macro ~0.50 refleja la dificultad inherente de distinguir clases adyacentes (B vs C, C vs D) sin información del GPA — variable excluida por data leakage. Este resultado honesto es más valioso que métricas infladas artificialmente.

## 4. Evaluación detallada del modelo seleccionado

In [ ]:
best_model = models['LogisticRegression']
y_pred_best = best_model.predict(X_test_sc)

print('=== Reporte por clase ===')
print(classification_report(y_test, y_pred_best, target_names=['A','B','C','D','F']))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A','B','C','D','F'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusión — LogisticRegression (Conjunto de Prueba)')
plt.tight_layout()
plt.savefig('../docs/matriz_confusion.png', dpi=120)
plt.show()

In [ ]:
# Importancia de variables (coeficientes de regresión logística)
feature_names = list(X.columns)
coef_mean = np.abs(best_model.coef_).mean(axis=0)
feat_imp = pd.Series(coef_mean, index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot(kind='barh', color='#3498db', ax=ax)
ax.set_title('Importancia de Variables (|coeficiente| promedio por clase)')
ax.set_xlabel('Importancia media')
plt.tight_layout()
plt.savefig('../docs/importancia_variables.png', dpi=120)
plt.show()

## 5. Serialización del modelo

In [ ]:
Path('../models').mkdir(parents=True, exist_ok=True)

# Serializar modelo y scaler
joblib.dump(best_model, '../models/modelo_final.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Verificar carga
modelo_cargado = joblib.load('../models/modelo_final.pkl')
print('Modelo cargado correctamente:', type(modelo_cargado))

# Guardar metadata
best_res = results['LogisticRegression']
metadata = {
    'modelo': 'LogisticRegression',
    'version': '1.0',
    'fecha_entrenamiento': '2026-06-10',
    'metrica_principal': 'f1_score_macro',
    'valor_metrica': best_res['f1_macro'],
    'accuracy': best_res['accuracy'],
    'precision_macro': best_res['precision_macro'],
    'recall_macro': best_res['recall_macro'],
    'variables_entrada': feature_names,
    'variable_objetivo': 'GradeClass',
    'clases': {'0':'A (Excelente)','1':'B (Bueno)','2':'C (Promedio)','3':'D (Bajo)','4':'F (Reprobado)'},
    'nota_gpa': 'GPA excluido (correlacion -0.97 con target = data leakage)',
    'observaciones': 'Comparados: RandomForest, GradientBoosting, LogisticRegression. Semilla 42.'
}

with open('../models/model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Metadata guardado en models/model_metadata.json')
print(json.dumps(metadata, indent=2, ensure_ascii=False))